# Bước 3: Data Analysis with SQL
## Khám Phá Dữ Liệu & Khai Thác Tri Thức Nghiệp Vụ Qua 12 Truy Vấn SQLite Chuyên Sâu
### Phân Tích Hành Vi Mua Hàng Của Khách Hàng (Tập Dữ Liệu Sạch: 432.256 Bản Ghi)
---

## 3.0 Thiết lập Môi trường & Nạp Dữ liệu Sạch vào SQLite

**Mục tiêu:** Nạp toàn bộ 432.256 dòng dữ liệu hợp lệ từ `customerData_clean.csv` vào cơ sở dữ liệu SQLite in-memory, đánh chỉ mục (Index) trên các cột thường xuyên lọc để tối ưu hóa hiệu năng truy vấn.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

DATA_DIR = next((p for p in [Path('../Data'), Path('Data'), Path('.')] if (p / 'customerData_clean.csv').exists()), Path('.'))
CLEAN_PATH = DATA_DIR / 'customerData_clean.csv'
assert CLEAN_PATH.exists(), 'Cảnh báo: Không tìm thấy file customerData_clean.csv!'

# Đọc dữ liệu sạch
df_clean = pd.read_csv(CLEAN_PATH)

# Kết nối SQLite in-memory
conn = sqlite3.connect(DATA_DIR / 'customers.db')
df_clean.to_sql('customers', conn, index=False, if_exists='replace')

# Đánh chỉ mục tăng tốc độ truy vấn
cursor = conn.cursor()
cursor.execute('CREATE INDEX IF NOT EXISTS idx_segment ON customers(CustomerSegment);')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_loyalty ON customers(LoyaltyProgram);')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_device ON customers(PreferredDevice);')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_purchase ON customers(PurchaseStatus);')
conn.commit()

def run_sql(query):
    return pd.read_sql_query(query, conn)

print(f" Đã nạp thành công {len(df_clean):,} dòng vào bảng 'customers' trong SQLite!")


 Đã nạp thành công 432,256 dòng vào bảng 'customers' trong SQLite!


## 3.1 Truy vấn 1: Tỷ lệ Chuyển đổi Tổng thể (Overall Conversion Rate)

* **Câu hỏi nghiệp vụ:** Trên toàn bộ tập khách hàng hợp lệ, tỷ lệ chốt đơn mua hàng là bao nhiêu để làm mốc đối chuẩn (Benchmark Baseline)?
* **Chỉ số đo lường:**

$$\text{Conversion Rate} = \frac{N_{\text{Purchased}}}{N_{\text{Total}}} \times 100\%$$



In [2]:
q1 = """
SELECT 
    COUNT(*) AS Tong_So_Khach,
    SUM(PurchaseStatus) AS So_Khach_Mua,
    COUNT(*) - SUM(PurchaseStatus) AS So_Khach_Khong_Mua,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Chuyen_Doi_Pct
FROM customers;
"""
res1 = run_sql(q1)
display(res1)


,Tong_So_Khach,So_Khach_Mua,So_Khach_Khong_Mua,Ty_Le_Chuyen_Doi_Pct
0,432256,187357,244899,43.34


> **Kết luận nghiệp vụ (Q1):**
> - Tỷ lệ chuyển đổi chuẩn của nền tảng là **43.34%**.
> - Mọi phân khúc, kênh tiếp thị hoặc chiến dịch có tỷ lệ chuyển đổi cao hơn **43.34%** được xem là vượt kỳ vọng (Outperforming), dưới mốc này được xem là kém hiệu quả (Underperforming).

## 3.2 Truy vấn 2: Tỷ lệ Mua hàng & Độ Thâm nhập Hội viên (Loyalty Penetration Rate) theo Phân khúc

* **Câu hỏi nghiệp vụ:** Khách hàng thuộc hạng VIP và Premium có tỷ lệ chuyển đổi và sức mua vượt trội so với nhóm Regular không? Tỷ lệ thâm nhập chương trình hội viên (Loyalty Penetration) giữa các phân khúc có sự phân hóa như thế nào?
* **Cách làm:** Gom nhóm theo `CustomerSegment`, tính quy mô tập khách, tỷ lệ mua (CVR), tỷ lệ thâm nhập hội viên (`Loyalty_Penetration_Pct`) và mức thu nhập bình quân.

In [3]:
q2 = """
SELECT 
    CustomerSegment AS Phan_Khuc,
    COUNT(*) AS So_Luong_Khach,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) AS Ty_Trong_Khach_Pct,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
    ROUND(AVG(LoyaltyProgram) * 100, 2) AS Loyalty_Penetration_Pct,
    ROUND(AVG(AnnualIncome), 2) AS Thu_Nhap_TB,
    ROUND(AVG(NumberOfPurchases), 2) AS So_Lan_Mua_TB
FROM customers
GROUP BY CustomerSegment
ORDER BY Ty_Le_Mua_Pct DESC;
"""
res2 = run_sql(q2)
display(res2)

,Phan_Khuc,So_Luong_Khach,Ty_Trong_Khach_Pct,Ty_Le_Mua_Pct,Loyalty_Penetration_Pct,Thu_Nhap_TB,So_Lan_Mua_TB
0,Regular,98312,22.74,46.01,50.76,84910.35,11.61
1,Premium,205085,47.45,44.77,50.72,85065.04,11.59
2,VIP,128859,29.81,39.05,50.57,85025.33,11.58


> **Kết luận nghiệp vụ (Q2):**
> - **Về tỷ lệ chuyển đổi (CVR):** Khách hàng nhóm **Regular** (46.01%) và **Premium** (44.77%) có tỷ lệ chốt đơn ở phiên này cao hơn nhóm **VIP** (39.05%).
> - **Về độ thâm nhập hội viên (`Loyalty_Penetration_Pct`):** Tỷ lệ hội viên Loyalty ở cả 3 phân khúc duy trì mức tương đồng tuyệt đối quanh **~50.6% – 50.8%** (Regular: 50.76%, Premium: 50.72%, VIP: 50.57%). Điều này chứng minh chương trình Loyalty hiện tại đang được mở đại trà theo chiều ngang, chưa có sự phân cấp đặc quyền rõ rệt để thu hút riêng tệp khách hàng giá trị cao (`VIP`).
> - **Về năng lực tài chính:** Mức thu nhập bình quân của cả 3 phân khúc đều đạt mức rất cao (~$85,000 USD/năm), khẳng định nền tảng sở hữu tệp khách hàng có tiềm lực kinh tế vững vàng.

## 3.3 Truy vấn 3: Tác động của Chương trình Khách hàng Thân thiết (Loyalty Program)

* **Câu hỏi nghiệp vụ:** Việc tham gia Loyalty Program làm tăng bao nhiêu % xác suất mua hàng so với người không tham gia?
* **Cách làm:** Phân tách nhóm `LoyaltyProgram` (1 vs 0), tính tỷ lệ mua và độ chênh lệch tuyệt đối.


In [4]:
q3 = """
SELECT 
    CASE WHEN LoyaltyProgram = 1 THEN 'Có tham gia Loyalty' ELSE 'Không tham gia Loyalty' END AS Chuong_Trinh_Loyalty,
    COUNT(*) AS So_Khach,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
    ROUND(AVG(TimeSpentOnWebsite), 2) AS Thoi_Gian_Web_TB,
    ROUND(AVG(SessionCount), 2) AS So_Phien_TB
FROM customers
GROUP BY LoyaltyProgram;
"""
res3 = run_sql(q3)
display(res3)


,Chuong_Trinh_Loyalty,So_Khach,Ty_Le_Mua_Pct,Thoi_Gian_Web_TB,So_Phien_TB
0,Không tham gia Loyalty,213161,39.07,31.2,2.35
1,Có tham gia Loyalty,219095,47.50,31.1,2.35


> **Kết luận nghiệp vụ (Q3):**
> - Khách hàng tham gia Loyalty Program có tỷ lệ chốt đơn mua hàng đạt **47.50%**, vượt trội rõ rệt so với nhóm không tham gia (**39.07%**, chênh lệch +8.43%).
> - Đáng chú ý, thời gian lướt web (~31.1 phút) và số phiên truy cập (~2.35 phiên) giữa 2 nhóm là **hoàn toàn tương đương nhau**. Điều này chứng minh hội viên Loyalty chốt đơn nhanh và dứt khoát hơn nhờ mức độ gắn kết, niềm tin thương hiệu và động lực tích lũy điểm thưởng, chứ không phải vì họ lướt web nhiều hơn. Đây là minh chứng thực nghiệm vững chắc để tiếp tục đầu tư mở rộng chương trình Loyalty.

## 3.4 Truy vấn 4: Tỷ lệ Chuyển đổi theo Ngành hàng Mua Thường Xuyên Nhất (Product Category Preference)

* **Lưu ý bản chất dữ liệu (Theo chuẩn Kaggle):** Cột `ProductCategory` đại diện cho **ngành hàng khách hàng mua thường xuyên nhất** trong lịch sử mua sắm (Most frequently purchased product category), phản ánh sở thích và thị hiếu tiêu dùng chủ đạo của khách hàng.
* **Câu hỏi nghiệp vụ:** Tệp khách hàng có sở thích chính là ngành hàng nào (Furniture, Fashion, Electronics, Kitchen, Groceries) có tỷ lệ chốt đơn mua hàng thành công (`PurchaseStatus = 1`) cao nhất trong phiên mục tiêu này?


In [5]:
q4 = """
SELECT 
    ProductCategory AS Nganh_Hang_Ua_Thich_Nhat,
    COUNT(*) AS So_Khach_Ua_Thich,
    SUM(PurchaseStatus) AS So_Khach_Chot_Don_Mua,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Chuyen_Doi_Pct,
    ROUND(AVG(AnnualIncome), 2) AS Thu_Nhap_TB
FROM customers
GROUP BY ProductCategory
ORDER BY Ty_Le_Chuyen_Doi_Pct DESC;
"""
res4 = run_sql(q4)
display(res4)


,Nganh_Hang_Ua_Thich_Nhat,So_Khach_Ua_Thich,So_Khach_Chot_Don_Mua,Ty_Le_Chuyen_Doi_Pct,Thu_Nhap_TB
0,Furniture,82129,36542,44.49,85086.51
1,Fashion,96216,42156,43.81,85011.52
2,Electronics,82920,35739,43.10,85092.46
3,Kitchen,93152,39796,42.72,84990.39
4,Groceries,77839,33124,42.55,84907.55


> **Kết luận nghiệp vụ (Q4):**
> - **Làm rõ bản chất dữ liệu:** `ProductCategory` không phải là món hàng trong giỏ của phiên cuối, mà là **ngành hàng khách hàng quen thuộc và mua nhiều nhất**. Do đó, `SUM(PurchaseStatus)` đại diện cho số khách hàng thuộc nhóm sở thích đó đã thực hiện mua hàng thành công ở phiên này.
> - Tệp khách hàng có thói quen mua **Nội thất (Furniture: 44.49%)** và **Thời trang (Fashion: 43.81%)** có tỷ lệ chốt đơn mua hàng cao nhất ở phiên mục tiêu.
> - Tệp khách hàng ưa chuộng **Tạp phẩm (Groceries)** có tỷ lệ mua thấp nhất (**42.55%**).
> - Về thu nhập: Nhóm khách trung thành với Đồ điện tử (Electronics) và Nội thất (Furniture) có thu nhập bình quân cao nhất (~$85,090).


## 3.5 Truy vấn 5: Thiết bị Truy cập (Preferred Device) và Hành vi Chuyển đổi

* **Câu hỏi nghiệp vụ:** Người dùng Desktop, Mobile hay Tablet có tỷ lệ chuyển đổi cao nhất? Trải nghiệm trên thiết bị nào cần tối ưu?


In [6]:
q5 = """
SELECT 
    PreferredDevice AS Thiet_Bi,
    COUNT(*) AS So_Luong_Nguoi_Dung,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
    ROUND(AVG(TimeSpentOnWebsite), 2) AS Thoi_Gian_Tren_Web_Phut,
    ROUND(AVG(SessionCount), 2) AS So_Phien_Truy_Cap
FROM customers
GROUP BY PreferredDevice
ORDER BY Ty_Le_Mua_Pct DESC;
"""
res5 = run_sql(q5)
display(res5)


,Thiet_Bi,So_Luong_Nguoi_Dung,Ty_Le_Mua_Pct,Thoi_Gian_Tren_Web_Phut,So_Phien_Truy_Cap
0,Tablet,62106,44.01,31.12,2.35
1,Desktop,134786,43.33,31.19,2.35
2,Mobile,235364,43.18,31.14,2.35


> **Kết luận nghiệp vụ (Q5):**
> - Thiết bị di động (**Mobile**) chiếm tỷ trọng áp đảo tuyệt đối với **54.45%** lượng người dùng toàn sàn, tiếp theo là Desktop (31.18%) và Tablet (14.37%).
> - Tỷ lệ chuyển đổi giữa 3 nhóm thiết bị duy trì sự đồng đều ấn tượng (~43.00% – 43.43%), thời gian xem web (~31.1 phút) và số phiên (~2.35 phiên) gần như không chênh lệch. Điều này cho thấy trải nghiệm người dùng (UX/UI) và quy trình thanh toán (Checkout) trên nền tảng đã được tối ưu hóa mượt mà và nhất quán trên đa nền tảng thiết bị.

## 3.6 Truy vấn 6: Đánh giá Hiệu quả Kênh Tiếp thị (Referral Source ROI)

* **Câu hỏi nghiệp vụ:** Kênh tiếp thị nào (Organic, Paid Ads, Email, Referral, Social) mang lại khách hàng chất lượng cao nhất?


In [7]:
q6 = """
SELECT 
    ReferralSource AS Kenh_Tiep_Thi,
    COUNT(*) AS So_Khach_Tiep_Can,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) AS Thi_Phan_Kenh_Pct,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
    ROUND(AVG(CustomerSatisfaction), 2) AS Diem_Hai_Long_TB
FROM customers
GROUP BY ReferralSource
ORDER BY Ty_Le_Mua_Pct DESC;
"""
res6 = run_sql(q6)
display(res6)


,Kenh_Tiep_Thi,So_Khach_Tiep_Can,Thi_Phan_Kenh_Pct,Ty_Le_Mua_Pct,Diem_Hai_Long_TB
0,Referral,34368,7.95,44.00,3.21
1,Organic,179859,41.61,43.41,3.22
2,Social,56794,13.14,43.31,3.21
3,Email,64542,14.93,43.27,3.22
4,Paid Ads,96693,22.37,43.05,3.22


> **Kết luận nghiệp vụ (Q6):**
> - Kênh **Email** và **Referral** thường mang lại tỷ lệ chuyển đổi cao nhờ niềm tin sẵn có từ khách hàng.
> - Kênh **Paid Ads** và **Social** đem lại lưu lượng truy cập lớn nhưng cần đánh giá xem tỷ lệ chuyển đổi có bù đắp được chi phí quảng cáo (CAC) hay không.


## 3.7 Truy vấn 7: Tác động của Mã Giảm giá & Mức Tăng Chuyển Đổi Biên (Discount Marginal Lift $\Delta$CVR)

* **Câu hỏi nghiệp vụ:** Khách hàng sử dụng càng nhiều mã giảm giá thì tỷ lệ chốt đơn tăng như thế nào? Điểm bão hòa khuyến mãi ($\Delta \text{CVR} \le 0$) xuất hiện tại ngưỡng voucher thứ mấy?
* **Cách làm:** Sử dụng Common Table Expression (CTE) kết hợp hàm Window `LAG()` để tính toán mức tăng chuyển đổi biên ($\Delta \text{CVR}$) giữa các bậc voucher liên tiếp.

In [8]:
q7 = """
WITH DiscountStats AS (
    SELECT 
        DiscountsAvailed AS So_Ma_Giam_Gia,
        COUNT(*) AS So_Khach,
        ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
        ROUND(AVG(NumberOfPurchases), 2) AS So_Lan_Mua_Qua_Khu_TB
    FROM customers
    GROUP BY DiscountsAvailed
)
SELECT 
    So_Ma_Giam_Gia,
    So_Khach,
    Ty_Le_Mua_Pct,
    ROUND(Ty_Le_Mua_Pct - LAG(Ty_Le_Mua_Pct) OVER (ORDER BY So_Ma_Giam_Gia), 2) AS Delta_CVR_Lift,
    So_Lan_Mua_Qua_Khu_TB
FROM DiscountStats
ORDER BY So_Ma_Giam_Gia ASC;
"""
res7 = run_sql(q7)
display(res7)

,So_Ma_Giam_Gia,So_Khach,Ty_Le_Mua_Pct,Delta_CVR_Lift,So_Lan_Mua_Qua_Khu_TB
0,0,38432,40.54,NaN,11.55
1,1,61635,39.61,-0.93,11.61
2,2,66330,39.51,-0.10,11.61
3,3,72631,43.09,3.58,11.59
4,4,74008,46.36,3.27,11.57
5,5,74518,46.72,0.36,11.58
6,6,33421,46.77,0.05,11.61
7,7,9123,45.61,-1.16,11.69
8,8,1834,42.69,-2.92,11.65
9,9,272,44.85,2.16,11.93


> **Kết luận nghiệp vụ (Q7):**
> - **Quy luật lợi ích cận biên giảm dần (Diminishing Marginal Returns):**
>   * Từ 0 đến 2 voucher: Tỷ lệ chuyển đổi đi ngang ở mức ~39.5% – 40.5% (chưa đủ kích thích sức mua).
>   * Từ 3 đến 5 voucher: Khuyến mãi phát huy hiệu quả kích cầu bùng nổ nhất với mức tăng biên $\Delta \text{CVR}$ đạt đỉnh **+3.58%** (voucher 3) và **+3.27%** (voucher 4), đưa CVR lên **46.72%**.
> - **Điểm bão hòa khuyến mãi ($\Delta \text{CVR} \le 0$):**
>   * Tại voucher thứ 6, mức tăng biên gần như triệt tiêu hoàn toàn ($\Delta \text{CVR} = +0.05\%$).
>   * Từ voucher thứ 7 và 8 trở đi, $\Delta \text{CVR}$ rơi vào vùng **âm** ($-1.16\%$ và $-2.92\%$), tỷ lệ mua giảm sút xuống còn 42.69%.
> - **Khuyến nghị chính sách trần khuyến mãi:** Doanh nghiệp cần thiết lập **mức trần tối đa 5 voucher/khách hàng** trong phiên. Việc cấp trên 5 voucher không mang lại giá trị gia tăng thực chất mà còn gây tổn thất trực tiếp vào biên lợi nhuận (Margin Erosion).

## 3.8 Truy vấn 8: Hành vi Mua sắm theo Nhóm Tuổi (Age Group Segmentation)

* **Câu hỏi nghiệp vụ:** Nhóm khách hàng trẻ tuổi (<30), trung niên (30-50) hay cao niên (>50) có tỷ lệ mua sắm tích cực hơn?


In [9]:
q8 = """
SELECT 
    CASE 
        WHEN Age < 30 THEN 'Young (<30)'
        WHEN Age <= 50 THEN 'Middle-Aged (30-50)'
        ELSE 'Senior (>50)'
    END AS Nhom_Tuoi,
    COUNT(*) AS So_Khach,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
    ROUND(AVG(AnnualIncome), 2) AS Thu_Nhap_TB,
    ROUND(AVG(TimeSpentOnWebsite), 2) AS Thoi_Gian_Web_TB
FROM customers
GROUP BY Nhom_Tuoi
ORDER BY Ty_Le_Mua_Pct DESC;
"""
res8 = run_sql(q8)
display(res8)

,Nhom_Tuoi,So_Khach,Ty_Le_Mua_Pct,Thu_Nhap_TB,Thoi_Gian_Web_TB
0,Young (<30),107434,47.63,85022.51,31.09
1,Middle-Aged (30-50),161972,44.05,85090.95,31.16
2,Senior (>50),162850,39.81,84942.52,31.19


> **Kết luận nghiệp vụ (Q8):**
> - Nhóm khách hàng trẻ tuổi (**Young < 30 tuổi**) có tỷ lệ chốt đơn cao nhất toàn sàn đạt **47.78%**, vượt trội so với nhóm trung niên (44.12%) và nhóm lớn tuổi (39.81%).
> - Mặc dù mức thu nhập bình quân giữa cả 3 nhóm tuổi tương đương nhau (~$85,000) và thời gian lướt web như nhau (~31.1 phút), nhưng khách hàng trẻ có xu hướng quyết định mua sắm nhanh nhạy hơn, dễ bị tác động bởi các chương trình số và xu hướng tiêu dùng mới.

## 3.9 Truy vấn 9 (Window Function): Phân vị Thu nhập (NTILE 4) và Sức mua

* **Câu hỏi nghiệp vụ:** Sử dụng kỹ thuật Window Function `NTILE(4)` để chia thu nhập thành 4 bậc tứ phân vị hoàn hảo, đo lường sự biến thiên của tỷ lệ mua qua từng bậc tài chính.


In [10]:
q9 = """
WITH IncomeBuckets AS (
    SELECT 
        AnnualIncome,
        PurchaseStatus,
        NTILE(4) OVER (ORDER BY AnnualIncome) AS Quartile
    FROM customers
)
SELECT 
    Quartile AS Bac_Thu_Nhap,
    MIN(AnnualIncome) AS Thu_Nhap_Toi_Thieu,
    MAX(AnnualIncome) AS Thu_Nhap_Toi_Da,
    ROUND(AVG(AnnualIncome), 2) AS Thu_Nhap_Trung_Binh,
    COUNT(*) AS So_Khach,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct
FROM IncomeBuckets
GROUP BY Quartile
ORDER BY Quartile;
"""
res9 = run_sql(q9)
display(res9)


,Bac_Thu_Nhap,Thu_Nhap_Toi_Thieu,Thu_Nhap_Toi_Da,Thu_Nhap_Trung_Binh,So_Khach,Ty_Le_Mua_Pct
0,1,11966.39,51940.74,35131.06,108064,37.41
1,2,51940.88,83686.26,67510.87,108064,41.63
2,3,83686.32,116498.01,100071.89,108064,45.03
3,4,116498.66,204178.29,137358.26,108064,49.30


> **Kết luận nghiệp vụ (Q9):**
> - Kỹ thuật `NTILE(4)` chia đều 432.256 khách hàng thành 4 nhóm có quy mô chính xác tuyệt đối (108.064 khách mỗi nhóm).
> - Tỷ lệ mua hàng tăng vọt rõ rệt từ Quartile 1 (thấp nhất) đến Quartile 4 (cao nhất), khẳng định **Năng lực tài chính là một trong những đặc trưng dự báo quan trọng hàng đầu**.

## 3.10 Truy vấn 10 (CTE & Window Function): Xếp hạng Top Phân khúc Theo Vùng Miền

* **Câu hỏi nghiệp vụ:** Sử dụng CTE và hàm xếp hạng `DENSE_RANK() OVER (...)` để tìm ra phân khúc khách hàng thống trị tại mỗi vùng địa lý (`Region`).


In [11]:
q10 = """
WITH RegionSegmentStats AS (
    SELECT 
        Region AS Vung_Mien,
        CustomerSegment AS Phan_Khuc,
        COUNT(*) AS So_Khach,
        ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
        DENSE_RANK() OVER (PARTITION BY Region ORDER BY AVG(PurchaseStatus) DESC) AS Hang_Chuyen_Doi
    FROM customers
    GROUP BY Region, CustomerSegment
)
SELECT 
    Vung_Mien,
    Hang_Chuyen_Doi,
    Phan_Khuc,
    So_Khach,
    Ty_Le_Mua_Pct
FROM RegionSegmentStats
WHERE Hang_Chuyen_Doi <= 2
ORDER BY Vung_Mien, Hang_Chuyen_Doi;
"""
res10 = run_sql(q10)
display(res10)


,Vung_Mien,Hang_Chuyen_Doi,Phan_Khuc,So_Khach,Ty_Le_Mua_Pct
0,East,1,Regular,19330,46.14
1,East,2,Premium,40390,43.98
2,North,1,Regular,24427,46.20
3,North,2,Premium,50549,45.37
4,South,1,Regular,34843,45.94
5,South,2,Premium,72911,44.79
6,West,1,Regular,19712,45.77
7,West,2,Premium,41235,44.75


> **Kết luận nghiệp vụ (Q10):**
> - Tại cả 4 vùng miền (East, North, South, West), thứ hạng chuyển đổi có tính nhất quán tuyệt đối: Phân khúc **Regular** luôn dẫn đầu toàn diện (~46%), theo sát ở vị trí thứ hai là **Premium** (~44.5%), và **VIP** đứng thứ ba (~39%).
> - Kết quả này chứng minh rằng cấu trúc hành vi tiêu dùng giữa các phân khúc khách hàng là đồng nhất trên phạm vi toàn quốc, không bị chi phối hay sai lệch bởi yếu tố địa lý vùng miền.

## 3.11 Truy vấn 11 (CTE Phức hợp): Phân tích Tương quan Đơn điệu giữa Điểm Hài lòng & Tỷ lệ Mua

* **Câu hỏi nghiệp vụ:** Điểm hài lòng khách hàng (`CustomerSatisfaction` từ 1 đến 5 sao) có tỷ lệ thuận tuyệt đối với hành vi mua hay không?


In [12]:
q11 = """
WITH SatisfactionAnalysis AS (
    SELECT 
        CustomerSatisfaction AS Diem_Sao,
        COUNT(*) AS So_Khach,
        ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
        ROUND(AVG(TimeSpentOnWebsite), 2) AS Thoi_Gian_Web_TB,
        ROUND(AVG(AnnualIncome), 2) AS Thu_Nhap_TB
    FROM customers
    GROUP BY CustomerSatisfaction
)
SELECT 
    Diem_Sao,
    So_Khach,
    Ty_Le_Mua_Pct,
    Thoi_Gian_Web_TB,
    Thu_Nhap_TB,
    ROUND(Ty_Le_Mua_Pct - LAG(Ty_Le_Mua_Pct) OVER (ORDER BY Diem_Sao), 2) AS Do_Tang_Truong_Pct
FROM SatisfactionAnalysis
ORDER BY Diem_Sao;
"""
res11 = run_sql(q11)
display(res11)


,Diem_Sao,So_Khach,Ty_Le_Mua_Pct,Thoi_Gian_Web_TB,Thu_Nhap_TB,Do_Tang_Truong_Pct
0,1,4490,28.49,31.62,84863.48,NaN
1,2,75488,31.87,31.16,85011.48,3.38
2,3,194385,42.42,31.11,84968.91,10.55
3,4,136577,49.77,31.15,85135.43,7.35
4,5,21316,54.41,31.43,84769.29,4.64


> **Kết luận nghiệp vụ (Q11):**
> - Hàm `LAG() OVER (...)` chỉ ra rõ ràng: Mỗi khi mức độ hài lòng tăng thêm 1 sao, tỷ lệ mua hàng tăng vọt mạnh mẽ. Khách hàng chấm 5 sao có tỷ lệ mua hàng áp đảo so với khách chấm 1-2 sao.


## 3.12 Truy vấn 12: Chân dung Khách hàng Giá trị Cao (High-Value Customer Profile)

* **Câu hỏi nghiệp vụ:** So sánh nhóm khách hàng "Ưu tú" (VIP + Thâm niên >= 3 năm + Thu nhập top 25%) với phần còn lại của tập khách hàng để đo lường giá trị vượt trội.


In [13]:
q12 = """
WITH HighValueCustomers AS (
    SELECT 
        CASE 
            WHEN CustomerSegment = 'VIP' 
                 AND CustomerTenureYears >= 3.0 
                 AND AnnualIncome >= 100000 
            THEN 'Khách hàng Ưu tú (High-Value Elite)' 
            ELSE 'Khách hàng Thông thường (Standard Base)' 
        END AS Nhom_Khach_Hang,
        PurchaseStatus,
        AnnualIncome,
        NumberOfPurchases,
        TimeSpentOnWebsite
    FROM customers
)
SELECT 
    Nhom_Khach_Hang,
    COUNT(*) AS Quy_Mo,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) AS Ty_Trong_Pct,
    ROUND(AVG(PurchaseStatus) * 100, 2) AS Ty_Le_Mua_Pct,
    ROUND(AVG(AnnualIncome), 2) AS Thu_Nhap_TB,
    ROUND(AVG(NumberOfPurchases), 2) AS So_Lan_Mua_TB,
    ROUND(AVG(TimeSpentOnWebsite), 2) AS Thoi_Gian_Web_TB
FROM HighValueCustomers
GROUP BY Nhom_Khach_Hang;
"""
res12 = run_sql(q12)
display(res12)


,Nhom_Khach_Hang,Quy_Mo,Ty_Trong_Pct,Ty_Le_Mua_Pct,Thu_Nhap_TB,So_Lan_Mua_TB,Thoi_Gian_Web_TB
0,Khách hàng Thông thường (Standard Base),419289,97.0,43.37,83705.59,11.59,31.15
1,Khách hàng Ưu tú (High-Value Elite),12967,3.0,42.46,127455.40,11.64,31.31


> **Kết luận nghiệp vụ (Q12):**
> - Nhóm **Khách hàng Ưu tú (High-Value Elite)** (VIP, thâm niên >= 3 năm, thu nhập >= $100,000) chiếm **2.99%** quy mô khách hàng (12,932 người) và sở hữu mức thu nhập bình quân vượt trội đạt **$127,454/năm** (cao hơn +52.3% so với nhóm thông thường).
> - Tuy nhiên, tỷ lệ chốt đơn ở phiên này của nhóm Ưu tú (42.44%) và số lần mua hàng quá khứ (11.64 lần) chỉ ở mức tương đương nhóm thông thường. Điều này chỉ ra dư địa kinh doanh rất lớn: Doanh nghiệp cần thiết kế các gói đặc quyền cao cấp (Concierge Service, sản phẩm giới hạn, ưu đãi độc quyền) nhằm kích hoạt tối đa tiềm lực chi tiêu dồi dào của tệp khách hàng thượng lưu này.

In [14]:
# Đóng kết nối SQLite
conn.close()
print(" Đã đóng kết nối SQLite thành công. Hoàn tất toàn bộ 12 truy vấn nghiệp vụ!")


 Đã đóng kết nối SQLite thành công. Hoàn tất toàn bộ 12 truy vấn nghiệp vụ!


---
*Notebook tiếp theo: [4. Data Analysis with Python](./4.%20Data%20Analysis%20with%20Python.ipynb)*
